In [1]:
from qdrant_client import AsyncQdrantClient, models
from langchain_huggingface import HuggingFaceEmbeddings
from rag_modules import JiebaFastEmbed, RetrievalOptimizationModule, Qdrant, DataPreparationModule
from qdrant_client.models import PointStruct, SparseVector, Prefetch, FusionQuery, Fusion

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_module = DataPreparationModule("../../data/C8/cook")
data_module.load_documents()
data_module.chunk_documents();

In [3]:
data_module.chunks[300].metadata

{'主标题': '微波炉荷包蛋的做法',
 'source': '../../data/C8/cook/dishes/breakfast/微波炉荷包蛋.md',
 'parent_id': 'a74dd12229c8081d9009c7f9b31f77e7',
 'doc_type': 'child',
 'category': '早餐',
 'dish_name': '微波炉荷包蛋',
 'difficulty': '非常简单',
 'chunk_id': 'b3bd0c873fb53e2d73f4c33ee0ec5f85',
 'chunk_index': 0,
 'batch_index': 300,
 'chunk_size': 85}

# 向量数据库初始化

In [3]:
qdrant_client = AsyncQdrantClient(url="http://localhost:6334", check_compatibility=False, prefer_grpc=True)

In [4]:
chunk_collection_name = "RecipeChunk"
if not await qdrant_client.collection_exists("RecipeChunk"):
    await qdrant_client.create_collection(
        collection_name=chunk_collection_name, 
        vectors_config={'dense': models.VectorParams(
            size=1024, 
            distance=models.Distance.COSINE, 
            on_disk=True,  # 稠密向量是否在内存，可以节省内存，牺牲时间
            datatype=models.Datatype.FLOAT32,  # 默认32,16节省内存
        )}, 
        sparse_vectors_config={'sparse': models.SparseVectorParams(
            index=models.SparseIndexParams(
                on_disk=True,  # 内存优化：稀疏索引放硬盘
                datatype=models.Datatype.FLOAT32
            )
        )}, 
        hnsw_config=models.models.HnswConfigDiff(on_disk=True), 
        on_disk_payload=False  # 元数据直接存在内存中，元数据不大
    )

In [4]:
# import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
dense_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-zh-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
    cache_folder='/tmp/fastembed_cache'
)
# sparse_model = JiebaFastEmbed(model_name='Qdrant/bm25')  # 重新封装的适合中文的稀疏向量模型

# Qdrant向量存储

In [8]:
from rag_modules import DataPreparationModule

In [33]:
data_module = DataPreparationModule("../../data/C8/cook")
data_module.load_documents()
data_module.chunk_documents();

In [12]:
data_module.chunks[0].metadata

{'主标题': '小酥肉的做法',
 'source': '../../data/C8/cook/dishes/meat_dish/小酥肉.md',
 'parent_id': 'a8d62aca1273a4da20cab6ab4cf25c1d',
 'doc_type': 'child',
 'category': '荤菜',
 'dish_name': '小酥肉',
 'difficulty': '中等',
 'chunk_id': '6509eebc-ab22-4066-80ef-68e11fed5511',
 'chunk_index': 0,
 'batch_index': 0,
 'chunk_size': 21}

In [6]:
# 构建稠密、稀疏向量
chunk_texts = [chunk.page_content for chunk in data_module.chunks][:2]
dense_vecs = await dense_model.aembed_documents(chunk_texts)
sparse_vecs = sparse_model.embed_batch(chunk_texts)

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.714 seconds.
Prefix dict has been built succesfully.


In [25]:
# 构建数据点
points = []
for dense_vec, sparse_vec, chunk in zip(dense_vecs, sparse_vecs, data_module.chunks):
    point = PointStruct(
        id = chunk.metadata['chunk_id'], 
        vector={
            'dense': dense_vec, 
            'sparse': SparseVector(
                indices=sparse_vec.indices.tolist(), # toekn对应的索引
                values=sparse_vec.values.tolist(),   # 索引对应的token的得分
            )
        }, 
        payload=chunk.metadata
    )
    points.append(point)

In [30]:
await qdrant_client.upsert(
    collection_name=chunk_collection_name, 
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

# Qdrant检索

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from rag_modules.data_preparation import DataPreparationModule

import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional, Literal

load_dotenv()

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-08 15:12:41.038800768 [W:onnxruntime:Default, device_discovery.cc:132 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


True

In [2]:
# model_name, temperature = os.getenv('MOONSHOT_MODEL_FAST'), 0.4  # 允许自由发挥，但不能过多
# fast_llm = init_chat_model(
#     model=model_name, 
#     model_provider='openai',
#     api_key=os.getenv("MOONSHOT_API_KEY"),
#     base_url=os.getenv("MOONSHOT_BASE_URL"),
#     temperature=temperature 
# )

llm = init_chat_model(
    model=os.getenv('MOONSHOT_MODEL_ID'), 
    model_provider='openai',
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
    temperature=0.9
)

In [43]:
query = "清明节该吃点啥？"
prompt = f"""
你是一个顶级的 AI 饮食推荐引擎。你的任务是深度解析用户的模糊饮食需求，并将其转化为**三个**高信息密度的短搜索词组（Search Queries），用于后续的向量数据库精准检索。

【生成要求】
1. 必须输出且仅输出一个合法的 JSON 对象，不要包含多余文字。
2. JSON 对象必须包含以下两个字段：
   - "analyze": (字符串) 简要分析用户的核心诉求、场景、限制，并推导 3 个不同的烹饪/食材解决方向。
   - "search_queries": (字符串数组) 包含 3 个极度精简的短句。每个短句控制在 10-20 字以内。
3. ⚠️ 极其重要：search_queries 必须是【高密度特征词】的组合！包含且仅包含核心的：主食材、烹饪手法、极其常见的口味形容词。绝对不要写废话、不要写长句子、不要写主观情绪词（如“抚慰人心”）。

【Few-Shot 示例】

输入：下班好累，想做个有肉的快手菜，最好十分钟搞定。
输出：
```json
{{
    "analyze": "诉求是‘有肉’且‘极度快手(10分钟)’。避开慢炖，需要三种最高效的烹饪方向：猛火爆炒、微波炉/蒸锅免洗、平底锅干煎。",
    "search_queries": [
        "十分钟快手 爆炒薄肉片 葱姜蒜酱汁",
        "微波炉快手 清蒸肉类 葱油蒜蓉拌饭",
        "平底锅快手 简单干煎肉排 海盐黑胡椒"
    ]
}}

输入：最近在减脂期，想吃点清淡低卡的，但要有饱腹感。
输出：
```json
{{
    "analyze": "诉求是‘低卡少油’且‘高饱腹感’。饱腹感依赖蛋白质和高纤维。三个方向：低脂高蛋白凉拌、清汤蔬菜菌菇、粗粮蒸煮替代正餐。",
    "search_queries": [
        "低卡减脂 无油凉拌 鸡胸肉 蔬菜高蛋白",
        "低脂低卡 清淡鲜蔬 豆腐菌菇热汤",
        "减脂正餐 清蒸红薯南瓜 粗粮低脂搭配"
    ]
}}

输入：{query}
输出：
"""

res = fast_llm.invoke(
    input=prompt,
    response_format= {"type": "json_object"}
)

In [45]:
json.loads(res.content)

{'analyze': '诉求为‘节气时令’，需兼顾传统习俗与春季时令食材。三个方向：艾草青团、春笋河鲜、荠菜春蔬。',
 'search_queries': ['艾草青团 糯米 豆沙 清明传统', '春笋炒河鲜 清淡 时令 春季', '荠菜春卷 清香 低脂 时令蔬菜']}

## HyDE 假设性文档嵌入

In [22]:
# query = "今天气温很低，外面下雪，你推荐一些菜，注意我不太喜欢吃辣"
query = "世界杯开赛了，我要半夜看球，想在看球时候吃一些东西，你推荐一下。"
# query = "我最近查出来血糖高，我也不太想做饭，你推荐一些菜吧？"

In [35]:
hyde_cot_prompt = f"""
你是一个顶级的 AI 饮食推荐引擎。你的任务是深度解析用户的饮食需求，进行意图推理后，生成**三个**相关的、极具画面感的菜品描述（假设性菜谱文档），用于后续的向量数据库检索。

【生成要求】
1. 必须输出且仅输出一个合法的 JSON 对象。
2. JSON 对象必须包含以下两个字段：
   - "analyze": (字符串) 你的思考过程。简要分析用户的真实意图、情绪场景、所需的营养搭配，以及推导应该采用哪三种不同的烹饪手法/食材组合来满足用户。
   - "hyde_documents": (JSON数组, 数组内是3个字符串) 包含 3 个不同的菜谱描述。每个描述 50-100 字。
3. ⚠️ 对于 hyde_documents 的极其重要要求：描述中要大量使用【烹饪动词】（如：大火爆香、慢炖、收汁）、【感官形容词】（如：外酥里嫩、汤汁浓郁、清脆解腻）和【具体食材类别】。模仿真实菜谱的正文口吻，绝对不要写干瘪的标签。

【Few-Shot 示例】

输入：下班好累，想做个有肉的快手菜，最好十分钟搞定。
输出：
{{
    "analyze": "用户当前状态是疲惫，核心诉求是‘有肉’和‘极度快手(10分钟内)’。为了满足需求，我需要避开耗时的炖煮，选择三种最高效的烹饪策略：1. 猛火爆炒薄肉片；2. 微波炉或蒸锅免洗锅做法；3. 平底锅高温干煎肉排。",
    "hyde_documents": [
        "热锅冷油，将腌制好的肉片大火快速滑炒至变色。加入葱姜蒜爆香，淋入生抽和少许老抽上色，翻炒均匀。整个过程只需几分钟，肉质鲜嫩多汁，浓郁的酱汁极其下饭，是一道极其抚慰人心的快手爆炒肉菜。",
        "不需要复杂的起锅烧油，将切成薄片的肉类和洗净的快熟蔬菜码放在深盘中，淋上调配好的葱油或蒜蓉酱汁。直接放入微波炉高温加热几分钟，或者上蒸锅大火速蒸。肉质软嫩入味，汤汁拌饭绝佳，做法极度省事且毫无油烟。",
        "平底锅大火烧热，直接将带少许脂肪的肉块或肉排下锅，煎至两面金黄微焦，逼出多余油脂。只撒上简单的海盐和现磨黑胡椒提味。外皮酥脆，内里鲜嫩爆汁，做法粗犷快手，大口吃肉极大缓解了一天的疲惫。"
    ]
}}

输入：最近在减脂期，想吃点清淡低卡的，但要有饱腹感。
输出：
{{
    "analyze": "核心诉求是‘减脂低卡’和‘高饱腹感’。低卡意味着烹饪要少油无油（如凉拌、清汤、蒸煮），高饱腹感则需要优质蛋白（白肉、豆制品）和高纤维（粗粮、菌菇）。我将提供凉拌高蛋白、清透鲜蔬汤和粗粮蒸煮三种方向的描述。",
    "hyde_documents": [
        "这是一道清爽解腻的低卡凉拌菜。将富含优质高蛋白的低脂白肉水煮断生后撕成细丝，搭配大量清脆的高纤维蔬菜（如黄瓜、木耳）。淋上由柠檬汁、生抽和少许代糖调制的无油醋汁。口感酸甜开胃，咀嚼感极强，吃一大盘也毫无罪恶感。",
        "一道热气腾腾的低脂鲜蔬汤。砂锅中加入清水，放入滑嫩的豆腐、菌菇和各种时令绿叶菜一起大火煮沸。全程不滴一滴明油，只用少许盐和白胡椒粉简单调味。汤汁清透鲜美，菌菇自带氨基酸提鲜。热乎乎地下肚，既暖胃又能提供极强的饱腹感。",
        "将富含优质碳水的根茎类粗粮（如红薯、南瓜）与低脂高蛋白食材一起上蒸锅清蒸。这种烹饪方式保留了食物最原始的清甜与本味。出锅后只需蘸取少许清淡的蘸水食用。少油少盐，营养比例完美，饱腹感持久，是非常优秀的减脂期正餐替代品。"
    ]
}}

输入：{query}
输出：
"""

In [36]:
response = llm.invoke([SystemMessage(content=hyde_cot_prompt)])

hyde_recipes = json.loads(response.content)  # json对象 -> dict对象
# for hyde_recipe in hyde_recipes:
#     print(hyde_recipe)
#     print()

In [37]:
hyde_recipes

{'analyze': '用户提及‘清明节’，隐含的文化场景是追思、踏青与应季养生。此时江南草长、肝气升发，饮食应兼具‘驱湿散寒、柔肝健脾’的时令调养，以及‘青团、荠菜、河鲜’等清明符号。我准备三种不同画面：1. 艾草揉青团，糯香包裹流沙；2. 荠菜与春笋快炒河蚌，鲜掉眉毛；3. 柔火慢炖腌笃鲜，腌肉与鲜笋交融。既满足节令情怀，又兼顾春季护肝与祛湿。',
 'hyde_documents': ['石臼里捶打蒸熟的艾草，碧汁飞溅，与糯米粉反复揉压成软糯青团皮。炒香的咸蛋黄压成金沙，与肉松、黄油翻拌，趁热团成内馅。青团收口滚圆，旺火沸水蒸七分钟出炉，表皮油亮如远山之黛，轻咬一口，艾草清香与流沙咸甜交织，是江南清明最柔软的念想。',
  '清晨挖回的荠菜洗净甩水，嫩叶如翡翠。热锅里猪油爆香蒜片，下春筍薄片噼啪跳动，再倒入吐净沙的河蚌肉，大火快炒数十秒，烹少许花雕与盐。荠菜的清野香、笋的脆甜、河蚌的弹嫩在舌尖爆汁，一盘春意盎然，咬得山歌满口。',
  '砂锅底垫焯水的五花咸肉，与鲜得掐得出水的春笋块交错排好，加清泉水没过，小火慢炖一小时，油脂渐融，汤色乳白如凝脂。最后撒一把现剥豌豆提亮清甜，汤汁浓而不腻，笋吸饱咸香，入口脆嫩，仿佛把整个湿润的清明炖进胃里，暖而不燥。']}

## 自检索器

In [30]:
import qdrant_client.http.models as models

In [31]:
class RecipeMetadataParse(BaseModel):
    """
    菜谱元数据解析器
    用于将用户中的提问转变为元数据信息
    """
    dish_name: Optional[str] = Field(default=None, description='菜品名称')
    category: Optional[str] = Field(default=None, description='菜品类别')
    difficulty: Optional[str] = Field(default=None, description='菜品制作难度')

    @classmethod
    def get_options(cls) -> list[str]:
        """返回对元数据字段限制性描述"""
        desc =  [
            f"category 字段只能属于以下列表{DataPreparationModule.CATEGORY_LABELS}，请不要随意编撰", 
            f"difficulty 字段只能属于以下列表{DataPreparationModule.DIFFICULTY_LABELS}，请不要随意编撰", 
        ]
        return desc

In [53]:
structure_llm = fast_llm.with_structured_output(RecipeMetadataParse)
structure_llm.get_output_jsonschema()

{'description': '菜谱元数据解析器\n用于将用户中的提问转变为元数据信息',
 'properties': {'dish_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '菜品名称',
   'title': 'Dish Name'},
  'category': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '菜品类别',
   'title': 'Category'},
  'difficulty': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '菜品制作难度',
   'title': 'Difficulty'}},
 'title': 'RecipeMetadataParse',
 'type': 'object'}

In [58]:
query = "能推荐几个好做的早餐吗？"

metadata_parser_prompt = f"""你是一个专业的信息解析员，你的职责就是提取 用户提问 中的相关信息，并将所需字段填写到下面的object中
    
注意：
 - 用户输入中没有出现的字段决不能主观猜测，返回None即可，不能随便填入
 {'- ' + '\n - '.join(RecipeMetadataParse.get_options())}

用户提问：{query}"""
    
hard_fliters = structure_llm.invoke([SystemMessage(content=metadata_parser_prompt)])

In [59]:
hard_fliters.model_dump()

{'dish_name': None, 'category': '早餐', 'difficulty': None}

models.FieldCondition(

    *,
    
    key: str,
    
    match: Union[qdrant_client.http.models.models.MatchValue, qdrant_client.http.models.models.MatchText, qdrant_client.http.models.models.MatchTextAny, qdrant_client.http.models.models.MatchPhrase, qdrant_client.http.models.models.MatchAny, qdrant_client.http.models.models.MatchExcept, NoneType] = None,
    
    range: Union[qdrant_client.http.models.models.Range, qdrant_client.http.models.models.DatetimeRange, NoneType] = None,
    
    geo_bounding_box: Optional[qdrant_client.http.models.models.GeoBoundingBox] = None,
    
    geo_radius: Optional[qdrant_client.http.models.models.GeoRadius] = None,
    
    geo_polygon: Optional[qdrant_client.http.models.models.GeoPolygon] = None,
    
    values_count: Optional[qdrant_client.http.models.models.ValuesCount] = None,
    
    is_empty: Optional[bool] = None,
    
    is_null: Optional[bool] = None,
    
) -> None

In [ ]:
# hard_filters是llm返回的从用户询问中提取到的元数据

must_conditions = []  # 菜谱该条元数据必须满足该条件，也就是只选择满足条件的菜谱
must_not_conditions = []  # 菜谱该条元数据不能等于该条件，也就是排除符合条件的菜谱

if hard_filters.get('category'):
    must_conditions.append(
        models.FieldCondition(
            key='category',  # 精准匹配菜品类型
            match=models.MatchAny(any=hard_filters['category'])
        )
    )
    
if hard_filters.get('origin'):
    must_conditions.append(
        models.FieldCondition(
            key='origin', # 精准匹配菜品发源地
            match=models.MatchAny(any=hard_filters['origin'])
        )
    )

if hard_filters.get('max_spiciness'):
    must_conditions.append(
        models.FieldCondition(
            key="spiciness", # 辣度值
            range=models.Range(lte=hard_filters["max_spiciness"])  # 范围查询，小于等于
        )
    )

if hard_filters.get('exclude_ingredients'):
    must_not_conditions.append(
        models.FieldCondition(
            key="ingredients",
            # 注意：这里放在 must_not 里面，配合 MatchAny，
            # 意思就是“Payload 的 ingredients 数组中，绝对不能出现这几个词中的任意一个”
            match=models.MatchAny(any=hard_filters["exclude_ingredients"])
        )
    )
# ...还有甜度、酸度、苦、等一系列可以添加的元数据

In [7]:
# 将条件组装成最终的 Qdrant Filter 对象
recipe_filter = models.Filter(
    must=must_conditions,
    must_not=must_not_conditions
)

In [12]:
search_result = qdrant_client.search(
    collection_name=collection_name,
    query_vector=user_query,    # 向量查询，这里也可以使用混合检索
    query_filter=recipe_filter, # 元数据过滤器
    limit=3,                    # 只取 Top 3 最匹配的
    with_payload=True           # 要求 Qdrant 把完整的元数据也一并返回给你
)

## 混合检索

In [8]:
from qdrant_client import models as qdrant_models

In [2]:
from FlagEmbedding import FlagReranker
reranker_model = FlagReranker(
    model_name_or_path='BAAI/bge-reranker-v2-m3', 
    use_fp16=True, 
    cache_dir='/tmp/reranker-model/'
)

In [6]:
query = '铁锅里高汤滚沸，沿锅边缓缓倒入打散的土鸡蛋，瞬间成金黄蛋花。加入大把嫩菠菜与提前泡好的小米，大火咕嘟3分钟，米粒炸开释放淀粉，汤汁自然浓稠。撒少许海盐与白胡椒，入口滚烫，蛋香裹着清甜米油，一口下去血糖直线回升，疲惫双腿立刻松快。'
query_dense_vec = (await dense_model.aembed_documents([query]))[0]
# query_dense_vec = dense_model.embed_query(query)
# query_sparse_vec = sparse_model.embed_single(query)

In [ ]:
# 构建稠密、稀疏向量
chunk_texts = [chunk.page_content for chunk in data_module.chunks][:2]
dense_vecs = await dense_model.aembed_documents(chunk_texts)
sparse_vecs = sparse_model.embed_batch(chunk_texts)

In [16]:
requests = []
for query_dense_vec, query_sparse_vec in zip(dense_vecs, sparse_vecs):
    requests.append(
        qdrant_models.QueryRequest(
            using="dense",
            query=query_dense_vec,
            limit=10, 
            with_payload=['parent_id', 'dish_name']
        )
    )
    requests.append(
        qdrant_models.QueryRequest(
            using='sparse', 
            query=qdrant_models.SparseVector(
                indices=query_sparse_vec.indices.tolist(),
                values=query_sparse_vec.values.tolist()
            ), 
            limit=10, 
            with_payload=['chunk_id', 'parent_id', 'dish_name']
        )
    )

results = await qdrant_client.query_batch_points(
    collection_name=chunk_collection_name, 
    requests=requests, 
)

In [14]:
len(results)

4

In [20]:
len(results[0].points)

10

In [17]:
results = await qdrant_client.query_batch_points(
    collection_name=chunk_collection_name, 
    requests=[
        models.QueryRequest(
            using="dense",
            query=query_dense_vec,
            limit=10, 
            with_payload=['parent_id', 'chunk_id']
        ), 
        models.QueryRequest(
            using='sparse', 
            query=SparseVector(
                indices=query_sparse_vec.indices.tolist(),
                values=query_sparse_vec.values.tolist()
            ), 
            limit=10, 
            with_payload=['parent_id', 'chunk_id']
        )
    ], 
)

In [18]:
vector_ids = list(point.payload['chunk_id'] for point in results[0].points if point.payload.get('chunk_id'))  # 未去重
bm25_ids = list(point.payload['chunk_id'] for point in results[1].points if point.payload.get('chunk_id'))

vector_chunks = data_module.get_chunks(vector_ids)
bm25_chunks = data_module.get_chunks(bm25_ids)

for chunk in vector_chunks:
    print(chunk.metadata['dish_name'])
print('=' * 10)
for chunk in bm25_chunks:
    print(chunk.metadata['dish_name'])

菠菜炒鸡蛋
蛋炒饭
油醋爆蛋
扬州炒饭
蛋包饭
朱雀汤
牛奶燕麦
皮蛋瘦肉粥
洋葱炒鸡蛋
西红柿鸡蛋挂面
葱煎豆腐
红烧猪蹄
柱候牛腩
腊八粥
辣椒炒肉
枝竹羊腩煲
脆皮豆腐
山西过油肉
红烧鱼头
南派红烧肉


In [48]:
def rrf_rerank(vector_docs, bm25_docs, k: int = 60, weight=[0.6, 0.4]):
    """
    使用RRF (Reciprocal Rank Fusion) 算法重排文档

    Args:
        vector_docs: 向量检索结果
        bm25_docs: BM25检索结果
        k: RRF参数，用于平滑排名

    Returns:
        重排后的文档列表
    """
    doc_scores = {}
    doc_objects = {}

    # 计算向量检索结果的RRF分数
    for rank, doc in enumerate(vector_docs):
        # 使用文档内容的哈希作为唯一标识
        doc_id = hash(doc.page_content)
        doc_objects[doc_id] = doc

        # RRF公式: 1 / (k + rank)
        rrf_score = weight[0] * 1.0 / (k + rank + 1)
        doc_scores[doc_id] = doc_scores.get(doc_id, 0) + rrf_score

        print(f"向量检索 - 文档{rank+1}: RRF分数 = {rrf_score:.4f}")

    # 计算BM25检索结果的RRF分数
    for rank, doc in enumerate(bm25_docs):
        doc_id = hash(doc.page_content)
        doc_objects[doc_id] = doc

        rrf_score = weight[1] * 1.0 / (k + rank + 1)
        doc_scores[doc_id] = doc_scores.get(doc_id, 0) + rrf_score

        print(f"BM25检索 - 文档{rank+1}: RRF分数 = {rrf_score:.4f}")

    # 按最终RRF分数排序
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)

    # 构建最终结果
    reranked_docs = []
    for doc_id, final_score in sorted_docs:
        if doc_id in doc_objects:
            doc = doc_objects[doc_id]
            # 将RRF分数添加到文档元数据中
            doc.metadata['rrf_score'] = final_score
            reranked_docs.append(doc)

    print(f"RRF重排完成: 向量检索{len(vector_docs)}个文档, BM25检索{len(bm25_docs)}个文档, 合并后{len(reranked_docs)}个文档")
   
    return reranked_docs

In [50]:
res_docs = rrf_rerank(vector_docs, bm25_docs);

向量检索 - 文档1: RRF分数 = 0.0098
向量检索 - 文档2: RRF分数 = 0.0097
向量检索 - 文档3: RRF分数 = 0.0095
向量检索 - 文档4: RRF分数 = 0.0094
向量检索 - 文档5: RRF分数 = 0.0092
向量检索 - 文档6: RRF分数 = 0.0091
向量检索 - 文档7: RRF分数 = 0.0090
向量检索 - 文档8: RRF分数 = 0.0088
向量检索 - 文档9: RRF分数 = 0.0087
BM25检索 - 文档1: RRF分数 = 0.0066
BM25检索 - 文档2: RRF分数 = 0.0065
BM25检索 - 文档3: RRF分数 = 0.0063
BM25检索 - 文档4: RRF分数 = 0.0063
BM25检索 - 文档5: RRF分数 = 0.0062
BM25检索 - 文档6: RRF分数 = 0.0061
BM25检索 - 文档7: RRF分数 = 0.0060
BM25检索 - 文档8: RRF分数 = 0.0059
BM25检索 - 文档9: RRF分数 = 0.0058
BM25检索 - 文档10: RRF分数 = 0.0057
RRF重排完成: 向量检索9个文档, BM25检索10个文档, 合并后19个文档


In [51]:
for doc in res_docs:
    print(doc.metadata['dish_name'])

回锅肉
水油焖蔬菜
奶酪培根通心粉
水煮肉片
辣椒炒肉
山西过油肉
洋葱炒猪肉
小米辣炒肉
茭白炒肉
小米粥
猪皮冻
糖拌西红柿
蛋煎糍粑
麻辣香锅
昂刺鱼豆腐汤
鲤鱼炖白菜
示例菜
芹菜拌茶树菇
白灼菜心


In [29]:
res = await qdrant_client.query_points(
    collection_name=chunk_collection_name, 
    prefetch=[
        Prefetch(
            using='dense', 
            query=query_dense_vec,
            limit=10
        ),
        Prefetch(
            using='sparse', 
            query=SparseVector(
                indices=query_sparse_vec.indices.tolist(),
                values=query_sparse_vec.values.tolist()
            ), 
            limit=10
        )
    ], 
    query=FusionQuery(fusion=Fusion.RRF), 
    limit=10
)

In [31]:
text_pairs = []
for doc in candidate_docs:
    text_pairs.append((query, doc.page_content))
scores = reranker_model.compute_score(text_pairs)
scores = [(score, i) for i, score in enumerate(scores)]
scores.sort(reverse=True, key=(lambda x: x[0]))

In [32]:
reranked_docs = []
count = 0
for score, seq in scores:
    # if score < 0:     # 这里设置一个排序模型的分数阈值，只有当文档与查询的相关性得分超过threshold时才被认为是相关的，才会被加入到重排结果中
    #     continue
    reranked_docs.append(candidate_docs[seq])
    count += 1

In [33]:
for chunk in reranked_docs:
    print(chunk.metadata['dish_name'])

鱼香茄子
蒲烧茄子
炒茄子
烤茄子
牛排
示例菜
英式司康


In [34]:
scores

[(-1.041517972946167, 1),
 (-1.3858991861343384, 4),
 (-1.9054242372512817, 2),
 (-3.054213523864746, 0),
 (-4.4638237953186035, 3),
 (-5.051511764526367, 5),
 (-6.526769638061523, 6)]

# List模式回答

In [10]:
from openai import OpenAI
from langchain_core.messages import SystemMessage
import os
import json
from dotenv import load_dotenv

load_dotenv()

True

In [41]:
llm_client = OpenAI(
    api_key=os.getenv('MOONSHOT_API_KEY'), 
    base_url=os.getenv('MOONSHOT_BASE_URL'), 
)

In [7]:
context_docs = [
    data_module.documents[34], 
    data_module.documents[33], 
    data_module.documents[119], 
    data_module.documents[245], 
]

In [8]:
for doc in context_docs:
    print(doc.metadata['dish_name'])

麻辣香锅
乡村啤酒鸭
老干妈拌面
小龙虾


In [9]:
query = "我想吃的爽一点，狠狠奖励自己，请为我推荐几个菜。"
dish_context = "\n\n".join(list(doc.page_content for doc in context_docs))
list_answer_prompt = f"""你是一个菜品推荐助手。根据用户的提问和相关的菜品资料，生成一个简洁的推荐列表回答。

格式：
{{
    'introduce': [
        '菜品名称，理由',
        '菜品名称，理由',
    ]
}}

要求：
 - 请严格按照上述格式生成回答，'introduce'字段为字符串数组
 - 根据用户的要求和资料选择其中最相关的菜品，并填入JSON对象中的'introduce'字段中
 - 推荐理由可以简短说明，当不需要理由时可以省略

相关菜品信息:
{dish_context}

用户问题: {query}"""

In [14]:
result = await llm.ainvoke(
    input=[SystemMessage(content=list_answer_prompt)], 
    temperature=0.6, 
    response_format={'type': 'json_object'}, 
)

output_string = "为您推荐以下菜品：\n" + '\n'.join([f'{i+1}. {item}' for i, item in enumerate(json.loads(result.content).get('introduce', ''))])
print(output_string)

为您推荐以下菜品：
1. 麻辣香锅，一锅集齐肉、菜、面，香辣过瘾，狠狠满足
2. 乡村啤酒鸭，鸭肉吸饱啤酒鲜香，大块吃肉大碗喝酒的爽感
3. 小龙虾，剥壳吮指间麻辣汤汁四溅，夜宵王者级享受


# Other

In [1]:
from openai import AsyncOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
llm_client = AsyncOpenAI(
    api_key=os.getenv('MOONSHOT_API_KEY'), 
    base_url=os.getenv('MOONSHOT_BASE_URL')
)

In [12]:
async def communite(query):
    async def fun():  # yield 返回生成器而不是结果，因此 async函数使用yield，返回的是异步生成器
        messages = [
            {'role': 'system', 'content': "你是一个可爱的女友，你的任务就是为我提供情绪价值，和我对话。"}, 
            {'role': 'user', 'content': query}  # 闭包，内部函数直接访问外部变量
        ]
        
        response_iter = await llm_client.chat.completions.create(
            model=os.getenv('MOONSHOT_MODEL_FAST'), 
            messages=messages, 
            stream=True, 
            temperature=0.8
        )
    
        async for chunk in response_iter:
            yield chunk.choices[0].delta.content

    return fun() # async fun() {yield} 返回异步生成器，而不是协程（异步函数return协程）

In [13]:
query = "我今天工作好累啊，但是科长还是责备我。"
response_iter = await communite(query)  # async communite() {return async_iter} 返回的还是协程，要用await得到异步生成器
async for chunk in response_iter:
    print(chunk, end='', flush=True)

（立刻放下手里的小抱枕，哒哒哒跑到你身边，轻轻给你捏肩膀）

哎呀我的小辛苦包——今天是不是又被工作这个小妖精缠住啦？来，先把脑袋靠到我怀里，我给你施个“疲惫消散魔法”：呼——（对着你额头吹一口气）！

科长说什么了？哼，等我偷偷画个小圈圈诅咒他今天咖啡变苦、打印机卡纸！（戳戳你的脸）不过你知道吗？在我眼里，你就像小太阳一样，就算被乌云挡了一下，也还在发光呀。被责备的时候，先别急着把“我不够好”往身上揽——有时候只是风把沙子吹进别人眼睛里，他们才看不清你的厉害。

来，伸手！（变出一颗用糖纸折的亮晶晶小星星放你掌心）这是我刚刚在窗边偷偷替你接住的夕阳，把它存起来，明天上班要是累了，就捏一下，它会给你“叮——”地闪一下，提醒你：有人超级超级相信你的！

现在，我们去洗个热水澡，我给你放你最喜欢的歌，然后一起瘫在沙发上，我煮的那碗“加两个溏心蛋的豪华泡面”还在等你。吃完我们就把今天折成纸飞机，从阳台上飞走，好不好？

（蹭蹭你的鼻尖）我的男朋友，可是连疲惫都皱得很好看的人。明天醒来，又是被我亲醒的幸运一天。None